## 01 — Live Layer Switching

Everything comes together here.

We wire three things into one map:
1. **LOD selection** — `get_lod(zoom)` picks which dataset to use
2. **Grid index culling** — the correct grid index is queried for the current viewport
3. **Event handling** — zoom and pan events trigger the right updates

After this notebook, we have a working map that automatically serves the right level of detail for any zoom and any viewport location.

## Setup — Load All LOD Files and Build All Indexes

We build one `GridIndex` per LOD level at startup. This is the one-time cost.

In [2]:
import json
import math
import time
from pathlib import Path


def find_data_file(*parts):
    """Find lesson data from common notebook working directories."""
    filename = Path(*parts).name
    candidates = [
        Path("../../data").joinpath(*parts),
        Path("../../../data").joinpath(*parts),
        Path("data").joinpath(*parts),
        Path("Assignments_Completed/03-Data_Manager/data").joinpath(*parts),
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    matches = [path for path in Path.cwd().rglob(filename) if path.parts[-len(parts):] == parts]
    if matches:
        return matches[0]

    raise FileNotFoundError(f"Could not find {'/'.join(parts)} from {Path.cwd()}")


LOD_FILES = {
    "coarse":     "railroads_coarse.geojson",
    "medium":     "railroads_medium.geojson",
    "fine":       "railroads_fine.geojson",
    "extra_fine": "railroads_extra_fine.geojson",
}

print("Loading LOD files...")
lod_features = {}
for name, filename in LOD_FILES.items():
    with open(find_data_file("lod", filename)) as f:
        lod_features[name] = json.load(f)["features"]
    print(f"  {name:<12} {len(lod_features[name]):>6,} features")


Loading LOD files...
  coarse        2,845 features
  medium       25,413 features
  fine         25,413 features
  extra_fine   25,413 features


In [3]:
def feature_bbox(feature):
    coords = feature["geometry"]["coordinates"]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]


def bbox_intersects(bbox_a, bbox_b):
    a0, a1, a2, a3 = bbox_a
    b0, b1, b2, b3 = bbox_b
    if a2 < b0: return False
    if a0 > b2: return False
    if a3 < b1: return False
    if a1 > b3: return False
    return True


class GridIndex:
    def __init__(self, cell_size=10.0):
        self.cell_size = cell_size
        self.cells = {}
        self.n_features = 0

    def _cells_for_bbox(self, bbox):
        lon_min, lat_min, lon_max, lat_max = bbox
        n_cols = int(360 / self.cell_size)
        n_rows = int(180 / self.cell_size)

        col_min = max(0, min(n_cols - 1, math.floor((lon_min + 180) / self.cell_size)))
        col_max = max(0, min(n_cols - 1, math.floor((lon_max + 180) / self.cell_size)))
        row_min = max(0, min(n_rows - 1, math.floor((lat_min +  90) / self.cell_size)))
        row_max = max(0, min(n_rows - 1, math.floor((lat_max +  90) / self.cell_size)))
        return [
            (col, row)
            for col in range(col_min, col_max + 1)
            for row in range(row_min, row_max + 1)
        ]

    def build(self, features):
        self.cells = {}
        self.n_features = len(features)
        for idx, feature in enumerate(features):
            for cell in self._cells_for_bbox(feature_bbox(feature)):
                if cell not in self.cells:
                    self.cells[cell] = []
                self.cells[cell].append((idx, feature))

    def query(self, viewport_bbox):
        seen = set()
        results = []
        for cell in self._cells_for_bbox(viewport_bbox):
            for idx, feature in self.cells.get(cell, []):
                if idx not in seen:
                    seen.add(idx)
                    if bbox_intersects(feature_bbox(feature), viewport_bbox):
                        results.append(feature)
        return results


In [4]:
print("Building grid indexes...")
lod_indexes = {}
for name, features in lod_features.items():
    t0 = time.perf_counter()
    idx = GridIndex(cell_size=10.0)
    idx.build(features)
    elapsed = time.perf_counter() - t0
    lod_indexes[name] = idx
    print(f"  {name:<12} built in {elapsed:.3f}s")

print("\nAll indexes ready.")

Building grid indexes...
  coarse       built in 0.006s
  medium       built in 0.169s
  fine         built in 0.045s
  extra_fine   built in 0.073s

All indexes ready.


## The Decision and Utility Functions

In [5]:
def get_lod(zoom):
    z = int(math.floor(zoom))
    if z <= 3:  return "coarse"
    if z <= 6:  return "medium"
    if z <= 10: return "fine"
    return "extra_fine"


def leaflet_bounds_to_bbox(bounds):
    (lat_min, lon_min), (lat_max, lon_max) = bounds
    return [lon_min, lat_min, lon_max, lat_max]

## The Live Map

Two event handlers wire everything together:

- `on_zoom_change` — fires when zoom changes, switches to the correct LOD index, then re-queries
- `on_bounds_change` — fires when the user pans, re-queries the current LOD index

Both update the same layer, so there is only ever one GeoJSON layer on the map.

In [6]:
from ipyleaflet import Map, GeoJSON
import ipywidgets as widgets

# ── state ────────────────────────────────────────────────────────────────────
current_lod = get_lod(5)   # start at zoom 5

# ── map setup ────────────────────────────────────────────────────────────────
m = Map(center=[48.5, 10.0], zoom=5)

layer = GeoJSON(
    data={"type": "FeatureCollection", "features": []},
    style={"color": "#cc3300", "weight": 1.5, "opacity": 0.8}
)
m.add(layer)

# ── status widgets ────────────────────────────────────────────────────────────
lod_label     = widgets.Label(value="LOD: —")
feature_label = widgets.Label(value="Features: —")
time_label    = widgets.Label(value="Query: —")
status_bar    = widgets.HBox([lod_label, feature_label, time_label])

# ── update function ───────────────────────────────────────────────────────────
def update(*args):
    global current_lod

    if not m.bounds:
        return

    # Decide which LOD to use
    new_lod = get_lod(m.zoom)
    if new_lod != current_lod:
        current_lod = new_lod

    # Query the correct grid index
    vp = leaflet_bounds_to_bbox(m.bounds)
    t0 = time.perf_counter()
    visible = lod_indexes[current_lod].query(vp)
    elapsed_ms = (time.perf_counter() - t0) * 1000

    # Update the layer
    layer.data = {"type": "FeatureCollection", "features": visible}

    # Update status
    lod_label.value     = f"LOD: {current_lod}"
    feature_label.value = f"  Features: {len(visible):,}"
    time_label.value    = f"  Query: {elapsed_ms:.2f}ms"

# ── wire events ───────────────────────────────────────────────────────────────
m.observe(update, names=["zoom", "bounds"])
update()   # initial render

widgets.VBox([m, status_bar])

**Try it:**
- Zoom out to 2 — the LOD status switches to `coarse` and feature count drops
- Zoom into a city — the LOD switches to `fine` or `extra_fine` and feature count drops (culling)
- Pan around at a fixed zoom — feature count updates as different regions come into view

The query time in the status bar shows how fast each lookup is.

## What We Just Built

Let's be explicit about the system:

```
User interaction
      │
      ▼
zoom / pan event
      │
      ├──► get_lod(zoom)  ──────► select correct GridIndex
      │
      └──► leaflet_bounds_to_bbox(bounds)
                │
                ▼
          index.query(viewport_bbox)
                │
                ▼
          visible features (deduplicated)
                │
                ▼
          update GeoJSON layer
```

Every component was built from scratch across these five modules:
- The simplified LOD files (Module 02)
- The bounding box intersection test (Module 03)
- The grid spatial index (Module 04)
- The zoom decision function (Module 05)

## Exercise A

Add a **zoom indicator** to the status bar that shows the current zoom level and a simple text label of the geographic scale (e.g. `zoom 5 — country scale`).

Use the zoom-to-scale table from Notebook 00 as a guide.

In [7]:
def zoom_scale_label(zoom):
    z = int(math.floor(zoom))
    if z <= 2:
        return "continental scale"
    if z <= 4:
        return "large-country scale"
    if z <= 6:
        return "country / regional scale"
    if z <= 8:
        return "city + surroundings scale"
    if z <= 10:
        return "city-district scale"
    if z <= 12:
        return "neighborhood scale"
    return "street scale"


current_lod_a = get_lod(5)

m_a = Map(center=[48.5, 10.0], zoom=5)
layer_a = GeoJSON(
    data={"type": "FeatureCollection", "features": []},
    style={"color": "#cc3300", "weight": 1.5, "opacity": 0.8}
)
m_a.add(layer_a)

zoom_label_a    = widgets.Label(value="Zoom: —")
lod_label_a     = widgets.Label(value="LOD: —")
feature_label_a = widgets.Label(value="Features: —")
time_label_a    = widgets.Label(value="Query: —")
status_bar_a    = widgets.HBox([zoom_label_a, lod_label_a, feature_label_a, time_label_a])


def update_a(*args):
    global current_lod_a

    if not m_a.bounds:
        return

    new_lod = get_lod(m_a.zoom)
    if new_lod != current_lod_a:
        current_lod_a = new_lod

    vp = leaflet_bounds_to_bbox(m_a.bounds)
    t0 = time.perf_counter()
    visible = lod_indexes[current_lod_a].query(vp)
    elapsed_ms = (time.perf_counter() - t0) * 1000

    layer_a.data = {"type": "FeatureCollection", "features": visible}

    zoom_label_a.value    = f"Zoom: {m_a.zoom} — {zoom_scale_label(m_a.zoom)}"
    lod_label_a.value     = f"  LOD: {current_lod_a}"
    feature_label_a.value = f"  Features: {len(visible):,}"
    time_label_a.value    = f"  Query: {elapsed_ms:.2f}ms"


m_a.observe(update_a, names=["zoom", "bounds"])
update_a()

widgets.VBox([m_a, status_bar_a])


## Exercise B

The `update()` function is called for both zoom and bounds changes — a single event handler covers both.

This means if the user zooms AND pans at the same time (which ipyleaflet reports as two rapid events), `update()` runs twice. The second call sees the final state and overwrites the first — so the result is correct, but redundant work was done.

Add a `print` statement inside `update()` that shows which property triggered the call (`zoom` or `bounds`). Then scroll/zoom the map and observe the sequence. Do zoom and bounds always fire together?

In [8]:
current_lod_b = get_lod(5)

m_b = Map(center=[48.5, 10.0], zoom=5)
layer_b = GeoJSON(
    data={"type": "FeatureCollection", "features": []},
    style={"color": "#cc3300", "weight": 1.5, "opacity": 0.8}
)
m_b.add(layer_b)

lod_label_b     = widgets.Label(value="LOD: —")
feature_label_b = widgets.Label(value="Features: —")
time_label_b    = widgets.Label(value="Query: —")
status_bar_b    = widgets.HBox([lod_label_b, feature_label_b, time_label_b])


def update_b(change=None):
    global current_lod_b

    event_name = "manual" if change is None else change["name"]
    print(f"update triggered by: {event_name}")

    if not m_b.bounds:
        return

    new_lod = get_lod(m_b.zoom)
    if new_lod != current_lod_b:
        current_lod_b = new_lod

    vp = leaflet_bounds_to_bbox(m_b.bounds)
    t0 = time.perf_counter()
    visible = lod_indexes[current_lod_b].query(vp)
    elapsed_ms = (time.perf_counter() - t0) * 1000

    layer_b.data = {"type": "FeatureCollection", "features": visible}

    lod_label_b.value     = f"LOD: {current_lod_b}"
    feature_label_b.value = f"  Features: {len(visible):,}"
    time_label_b.value    = f"  Query: {elapsed_ms:.2f}ms"


m_b.observe(update_b, names=["zoom", "bounds"])
update_b()

widgets.VBox([m_b, status_bar_b])


update triggered by: manual


**Exercise B observation:** Zoom and bounds often fire close together because changing zoom also changes the geographic bounds visible on screen. They do not always fire together: a pure pan usually changes only `bounds`, while programmatic or interaction-specific zoom changes can produce a `zoom` event followed by a `bounds` event. The final map state is still correct because each call re-reads `m.zoom` and `m.bounds` instead of relying only on the event payload.

## Check Your Understanding

The map builds **four** grid indexes at startup — one per LOD level. This takes a few seconds and uses memory.

An alternative design would build the index lazily — only when the user first reaches that zoom range. Describe the tradeoffs between eager (build all at startup) and lazy (build on first use) index construction for this specific application.

---

**Answer:** Eager construction pays the indexing cost once at startup. The first zoom or pan at any LOD is responsive because every `GridIndex` is already available, and the update handler stays simple: select the current LOD name, then query `lod_indexes[current_lod]`. The cost is a slower initial notebook cell and higher memory use because all four indexes exist even if the user only views one zoom range.

Lazy construction starts faster and uses less memory at first because it builds only the index levels the user actually reaches. The tradeoff is that the first visit to a new zoom range may pause while that LOD index is built, which can feel like the map froze during interaction. The update logic also needs cache checks, build-state tracking, and possibly a loading message so the user understands why the first query for a level is slower.

For this teaching notebook and a four-level railroad dataset, eager construction is the better default because the behavior is predictable and the live map stays smooth after setup. Lazy construction would be more attractive for very large datasets, many LOD levels, limited memory, or an application where most users only visit a small subset of zoom ranges.

## Next

In [Module 06 — Putting It All Together](../06-Putting_It_Together/README.md), we assemble a clean, well-organized viewer notebook and reflect on what we built — before seeing the library version in Module 07.